# Spin-Flip ORMAS-CI (SF-ORMAS) Tutorial

This notebook demonstrates the spin-flip ORMAS-CI method for describing
multiconfigurational electronic states starting from a single-determinant
high-spin reference. We use a stretched H2 molecule and twisted ethylene
as test systems to illustrate:

- How to set up an ROHF reference for spin-flip calculations
- How to configure `SFORMASConfig` for single spin-flip
- How to run SF-ORMAS through PySCF's CASCI interface
- How to use spin diagnostics (⟨S²⟩) to identify singlet vs. triplet states
- How to compare SF-ORMAS against full CASCI

In [ ]:
import numpy as np
from pyscf import gto, scf, mcscf
from pyscf.ormas_ci import SFORMASFCISolver, SFORMASConfig, Subspace
from pyscf.ormas_ci.determinants import casci_determinant_count
from pyscf.ormas_ci.spinflip import count_sf_determinants

## Setting up a high-spin ROHF reference

SF-ORMAS starts from a high-spin ROHF reference. For singlet diradicals we
use a triplet (S=1, M_s=+1) ROHF. ROHF is required because it provides a
single set of spatial orbitals shared by alpha and beta electrons. UHF is
not appropriate since it gives different spatial orbitals for each spin,
which breaks the spin-flip formalism.

Here we build a stretched H2 molecule (2.0 Å bond length) in the 6-31G
basis. At this geometry the molecule has strong diradical character.

In [ ]:
mol = gto.M(atom='H 0 0 0; H 0 0 2.0', basis='6-31g', spin=2, verbose=0)
mf = scf.ROHF(mol)
mf.verbose = 0
mf.run()
print(f"ROHF energy: {mf.e_tot:.10f} Hartree")
print(f"Spin (2S): {mol.spin}")

## Configuring the spin-flip ORMAS space

`SFORMASConfig` specifies the reference spin, target spin, and subspaces.
For a single spin-flip targeting a singlet from a triplet: `ref_spin=2`
(triplet, 2S=2), `target_spin=0` (singlet, 2S=0), `n_spin_flips=1`.

The `nelecas_target` property gives the target-sector electron counts
(alpha, beta) that PySCF's CASCI needs. With one spin flip, an alpha
electron is converted to beta, so (2, 0) becomes (1, 1).

In [ ]:
sf_config = SFORMASConfig(
    ref_spin=2,        # triplet reference (2S = 2)
    target_spin=0,     # singlet target (2S = 0)
    n_spin_flips=1,
    n_active_orbitals=2,
    n_active_electrons=2,
    subspaces=[Subspace("sigma", [0, 1], 0, 4)],
)

print(f"Reference (alpha, beta): {sf_config.nelecas_reference}")
print(f"Target (alpha, beta):    {sf_config.nelecas_target}")
print(f"SF determinants:         {count_sf_determinants(sf_config)}")
print(f"Full CAS determinants:   {casci_determinant_count(2, sf_config.nelecas_target)}")

## Running the SF-ORMAS calculation

Plug `SFORMASFCISolver` into PySCF's CASCI as a drop-in FCI solver
replacement. Use the target-sector electron counts from `nelecas_target`.

In [ ]:
n_a, n_b = sf_config.nelecas_target
mc = mcscf.CASCI(mf, 2, (n_a, n_b))
mc.verbose = 0
mc.fcisolver = SFORMASFCISolver(sf_config)
e_sf = mc.kernel()[0]
print(f"SF-ORMAS energy: {e_sf:.10f} Hartree")

## Spin diagnostics

The M_s=0 sector contains both singlet and triplet states. We use the
expectation value ⟨S²⟩ to identify the spin of each state: ⟨S²⟩ ≈ 0
for a singlet (2S+1 = 1) and ⟨S²⟩ ≈ 2 for a triplet (2S+1 = 3).

In [ ]:
ss, mult = mc.fcisolver.spin_square(mc.ci, 2, (n_a, n_b))
print(f"<S^2> = {ss:.6f}")
print(f"2S+1  = {mult:.3f}")
if ss < 0.5:
    print("State identified as SINGLET")
else:
    print("State identified as TRIPLET")

## Multi-root calculation

Setting `nroots > 1` captures multiple states in the M_s=0 sector.
Both the singlet and triplet M_s=0 components are found, and we can
identify each state from its ⟨S²⟩ value.

In [ ]:
mc2 = mcscf.CASCI(mf, 2, (n_a, n_b))
mc2.verbose = 0
mc2.fcisolver = SFORMASFCISolver(sf_config)
mc2.fcisolver.nroots = 2
mc2.kernel()

print(f"{'Root':>4} {'Energy (Ha)':>16} {'<S^2>':>8} {'2S+1':>6} {'State':>10}")
print("-" * 50)
for i, e in enumerate(mc2.e_tot):
    ci = mc2.ci[i]
    ss, mult = mc2.fcisolver.spin_square(ci, 2, (n_a, n_b))
    state = "singlet" if ss < 0.5 else "triplet"
    print(f"{i:>4} {e:>16.10f} {ss:>8.4f} {mult:>6.2f} {state:>10}")

## Comparison with full CASCI

When SF-ORMAS subspace bounds are set wide enough to include all
configurations (as in our single-subspace setup above), the result
must match PySCF's built-in CASCI exactly.

In [ ]:
mc_ref = mcscf.CASCI(mf, 2, (n_a, n_b))
mc_ref.verbose = 0
e_ref = mc_ref.kernel()[0]

print("Energy Comparison")
print("=" * 45)
print(f"  PySCF CASCI:  {e_ref:.10f} Hartree")
print(f"  SF-ORMAS:     {e_sf:.10f} Hartree")
print(f"  Difference:   {abs(e_ref - e_sf):.2e} Hartree")

## Twisted ethylene: a larger example

Twisted ethylene (90° dihedral) has diradical character with
near-degenerate singlet and triplet states. The two electrons in the
pi and pi* orbitals become essentially non-bonding at the twisted
geometry. We run a multi-root SF-ORMAS to compute the singlet-triplet
gap.

In [ ]:
mol_eth = gto.M(
    atom="""
    C  0.000  0.000  0.000
    C  1.340  0.000  0.000
    H -0.500  0.930  0.000
    H -0.500 -0.930  0.000
    H  1.840  0.000  0.930
    H  1.840  0.000 -0.930
    """,
    basis="6-31g", spin=2, verbose=0,
)
mf_eth = scf.ROHF(mol_eth)
mf_eth.verbose = 0
mf_eth.run()
print(f"Ethylene ROHF energy: {mf_eth.e_tot:.10f} Hartree")

In [ ]:
sf_eth = SFORMASConfig(
    ref_spin=2, target_spin=0, n_spin_flips=1,
    n_active_orbitals=2, n_active_electrons=2,
    subspaces=[Subspace("pi", [0, 1], 0, 4)],
)

n_a, n_b = sf_eth.nelecas_target
mc_eth = mcscf.CASCI(mf_eth, 2, (n_a, n_b))
mc_eth.verbose = 0
mc_eth.fcisolver = SFORMASFCISolver(sf_eth)
mc_eth.fcisolver.nroots = 2
mc_eth.kernel()

print("Twisted Ethylene SF-ORMAS Results")
print("=" * 55)
print(f"{'Root':>4} {'Energy (Ha)':>16} {'<S^2>':>8} {'2S+1':>6} {'State':>10}")
print("-" * 55)
for i, e in enumerate(mc_eth.e_tot):
    ci = mc_eth.ci[i]
    ss, mult = mc_eth.fcisolver.spin_square(ci, 2, (n_a, n_b))
    state = "singlet" if ss < 0.5 else "triplet"
    print(f"{i:>4} {e:>16.10f} {ss:>8.4f} {mult:>6.2f} {state:>10}")

gap_ha = mc_eth.e_tot[1] - mc_eth.e_tot[0]
gap_mh = gap_ha * 1000
print(f"\nSinglet-triplet gap: {gap_mh:.4f} mHartree ({gap_ha * 627.509:.2f} kcal/mol)")

## Summary

In this tutorial we demonstrated the SF-ORMAS workflow:

1. Built molecules and ran ROHF for the high-spin reference state.
2. Configured `SFORMASConfig` with reference/target spin and subspace bounds.
3. Plugged `SFORMASFCISolver` into PySCF's CASCI as a drop-in solver.
4. Used spin diagnostics (⟨S²⟩) to identify singlet vs. triplet states.
5. Ran multi-root calculations to obtain excited states.
6. Verified that unrestricted SF-ORMAS reproduces CASCI exactly.
7. Applied SF-ORMAS to twisted ethylene to compute the singlet-triplet gap.

Key takeaways:
- SF-ORMAS starts from a single-determinant high-spin ROHF reference
- It targets the low-spin sector via spin-flip excitations
- ORMAS subspace bounds ensure spin completeness (⟨S²⟩ eigenstates)
- Multi-root calculations capture both singlet and triplet M_s=0 states